# Day 5 Task Sheet
**Topic:** Software Engineering Habits and Week 1 Deliverable

## Task 01: Write your first real tests
**What I did:** Wrote a small module `mathutils.py` with three functions (`average`, `biggest`, `is_prime`) and a test file `test_mathutils.py` with 12 tests, then ran them with pytest.

**Logic:**
- A test is a small function that checks one fact with `assert`. If the condition is true, the test passes. If it's false, the test fails and pytest tells me exactly what was different.
- For every function I wrote a **happy-path** test (normal input) and **edge-case** tests (empty list, single item, 0, 1, negative number), because edge cases are where bugs hide.
- I chose the behaviour for edge cases myself: `average([])` returns 0, and `biggest([])` returns `None`.
- `%%writefile` at the top of a cell saves the cell's contents into a file, so the notebook creates the real `.py` files.

In [24]:
%%writefile mathutils.py
"""Provide small math helpers: average, biggest and is_prime."""


def average(nums: list[float]) -> float:
    """Return the mean of a list of numbers.

    Args:
        nums: A list of numbers. May be empty.

    Returns:
        The mean of the numbers, or 0 if the list is empty.
    """
    if not nums:
        return 0
    return sum(nums) / len(nums)


def biggest(nums: list[int]) -> int | None:
    """Return the largest number in a list.

    Args:
        nums: A list of numbers. May be empty.

    Returns:
        The largest number, or None if the list is empty.
    """
    if not nums:
        return None
    return max(nums)


def is_prime(n: int) -> bool:
    """Return whether a number is prime.

    Args:
        n: The whole number to check.

    Returns:
        True if n is prime, otherwise False. Numbers below 2 (0, 1 and
        negatives) are never prime, so they return False.
    """
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

Overwriting mathutils.py


**How the functions work:**
- `average`: if the list is empty it returns 0, because dividing by zero would crash. Otherwise it returns the total divided by the count.
- `biggest`: returns `None` for an empty list (there is no biggest item), otherwise `max(nums)`.
- `is_prime`: numbers below 2 are never prime, so it returns `False` for 1, 0 and negatives. For others it tries every possible divisor from 2 up to the square root of `n`. If any divides evenly, the number is not prime. Checking only up to the square root is enough, because any factor larger than that would be paired with a smaller one.

In [25]:
%%writefile test_mathutils.py
from mathutils import average, biggest, is_prime


def test_average_basic():
    assert average([2, 4, 6]) == 4.0


def test_average_empty():
    assert average([]) == 0


def test_average_single():
    assert average([5]) == 5.0


def test_biggest_basic():
    assert biggest([3, 9, 2]) == 9


def test_biggest_empty():
    assert biggest([]) is None


def test_biggest_negatives():
    assert biggest([-5, -2, -9]) == -2


def test_is_prime_two():
    assert is_prime(2) is True


def test_is_prime_seven():
    assert is_prime(7) is True


def test_is_prime_composite():
    assert is_prime(9) is False


def test_is_prime_one():
    assert is_prime(1) is False


def test_is_prime_zero():
    assert is_prime(0) is False


def test_is_prime_negative():
    assert is_prime(-3) is False


Overwriting test_mathutils.py


| Function | Happy path | Edge cases |
|---|---|---|
| `average` | `[2, 4, 6]` gives 4.0 | empty list gives 0, single item `[5]` gives 5.0 |
| `biggest` | `[3, 9, 2]` gives 9 | empty list gives `None`, all-negative list gives -2 |
| `is_prime` | 7 is prime, 9 is not | 2 (the only even prime), 1, 0 and -3 are not prime |

In [26]:
import sys
!"{sys.executable}" -m pytest -v

============================= test session starts =============================
platform win32 -- Python 3.12.1, pytest-9.2.0, pluggy-1.7.0 -- c:\Users\gajen\OneDrive\Desktop\Codetrade_internship\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\gajen\OneDrive\Desktop\Codetrade_internship\Day5
plugins: platformdirs-4.14.7
collecting ... collected 12 items

test_mathutils.py::test_average_basic PASSED                             [  8%]
test_mathutils.py::test_average_empty PASSED                             [ 16%]
test_mathutils.py::test_average_single PASSED                            [ 25%]
test_mathutils.py::test_biggest_basic PASSED                             [ 33%]
test_mathutils.py::test_biggest_empty PASSED                             [ 41%]
test_mathutils.py::test_biggest_negatives PASSED                         [ 50%]
test_mathutils.py::test_is_prime_two PASSED                              [ 58%]
test_mathutils.py::test_is_prime_seven PASSED                   

**Deliberately breaking `average`:** I change `sum(nums) / len(nums)` to `sum(nums) / (len(nums) + 1)`. This looks almost right, which is how real bugs look. Then I run the tests again.

In [27]:
from pathlib import Path

p = Path("mathutils.py")
original = p.read_text()
broken = original.replace("sum(nums) / len(nums)", "sum(nums) / (len(nums) + 1)")
p.write_text(broken)
print("mathutils.py is now broken on purpose")

mathutils.py is now broken on purpose


In [28]:
!"{sys.executable}" -m pytest -v

============================= test session starts =============================
platform win32 -- Python 3.12.1, pytest-9.2.0, pluggy-1.7.0 -- c:\Users\gajen\OneDrive\Desktop\Codetrade_internship\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\gajen\OneDrive\Desktop\Codetrade_internship\Day5
plugins: platformdirs-4.14.7
collecting ... collected 12 items

test_mathutils.py::test_average_basic FAILED                             [  8%]
test_mathutils.py::test_average_empty PASSED                             [ 16%]
test_mathutils.py::test_average_single FAILED                            [ 25%]
test_mathutils.py::test_biggest_basic PASSED                             [ 33%]
test_mathutils.py::test_biggest_empty PASSED                             [ 41%]
test_mathutils.py::test_biggest_negatives PASSED                         [ 50%]
test_mathutils.py::test_is_prime_two PASSED                              [ 58%]
test_mathutils.py::test_is_prime_seven PASSED                   

**How pytest reports a failure:**
- `FAILED` next to the test names shows which tests went red. Only the two `average` tests failed, so the bug is in `average`.
- The line starting with `>` is the line that failed.
- The lines starting with `E` explain why: the test expected `4.0` but `average([2, 4, 6])` returned `3.0`. The `where` line shows the exact call and the value it gave.
- The summary at the bottom says `2 failed, 10 passed`. The other 10 tests were not affected, which shows the tests pin down each function separately.

In [29]:
p.write_text(original)
print("mathutils.py is fixed again")

mathutils.py is fixed again


In [30]:
!"{sys.executable}" -m pytest -v

============================= test session starts =============================
platform win32 -- Python 3.12.1, pytest-9.2.0, pluggy-1.7.0 -- c:\Users\gajen\OneDrive\Desktop\Codetrade_internship\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\gajen\OneDrive\Desktop\Codetrade_internship\Day5
plugins: platformdirs-4.14.7
collecting ... collected 12 items

test_mathutils.py::test_average_basic PASSED                             [  8%]
test_mathutils.py::test_average_empty PASSED                             [ 16%]
test_mathutils.py::test_average_single PASSED                            [ 25%]
test_mathutils.py::test_biggest_basic PASSED                             [ 33%]
test_mathutils.py::test_biggest_empty PASSED                             [ 41%]
test_mathutils.py::test_biggest_negatives PASSED                         [ 50%]
test_mathutils.py::test_is_prime_two PASSED                              [ 58%]
test_mathutils.py::test_is_prime_seven PASSED                   

**Why does the file have to be named `test_*.py` and the functions `test_*`?**
(Answer: pytest finds tests automatically by naming convention. It collects files named `test_*.py` and runs functions whose names start with `test_`. Other files and functions are ignored, so I never have to list the tests anywhere.)

**Is 1 prime? Is 2? Is 0?**
(Answer: 1 is not prime, because a prime has exactly two divisors and 1 has only one. 2 is prime, and it is the only even prime. 0 and negative numbers are not prime. My tests `is_prime(1)`, `is_prime(2)`, `is_prime(0)` and `is_prime(-3)` pin down each of these answers.)

**What does `pytest -v` show that plain `pytest` does not?**
(Answer: `-v` means verbose. It lists every test by name with `PASSED` or `FAILED`, while plain `pytest` only shows a row of dots. With `-v` I can see exactly which tests ran.)

## Task 02: Document it properly
**What I did:** Added a Google-style docstring to every function in `mathutils.py`, plus a one-line docstring for the whole module, then used `help(mathutils)` to confirm they show up.

**Logic:** a docstring is a string written as the first line inside a function (or at the top of a module). Python stores it, so tools like `help()` and the editor's hover text can show it.

Each function docstring has the same parts:
1. A **one-line summary** in the imperative ("Return the mean..."), saying what the function does.
2. An **Args** section listing each argument and what it means.
3. A **Returns** section saying what comes back.
4. **Edge-case behaviour** written in those sections: `average` gives 0 for an empty list, `biggest` gives `None`, and `is_prime` gives `False` for 0, 1 and negatives.

In [31]:
import importlib
import mathutils

importlib.reload(mathutils)
help(mathutils)

Help on module mathutils:

NAME
    mathutils - Provide small math helpers: average, biggest and is_prime.

FUNCTIONS
    average(nums: list[float]) -> float
        Return the mean of a list of numbers.

        Args:
            nums: A list of numbers. May be empty.

        Returns:
            The mean of the numbers, or 0 if the list is empty.

    biggest(nums: list[int]) -> int | None
        Return the largest number in a list.

        Args:
            nums: A list of numbers. May be empty.

        Returns:
            The largest number, or None if the list is empty.

    is_prime(n: int) -> bool
        Return whether a number is prime.

        Args:
            n: The whole number to check.

        Returns:
            True if n is prime, otherwise False. Numbers below 2 (0, 1 and
            negatives) are never prime, so they return False.

DATA
    Optional = typing.Optional
        Optional[X] is equivalent to Union[X, None].

FILE
    c:\users\gajen\onedrive\deskt

`help(mathutils)` shows the module docstring next to the module name, and each function's summary, Args and Returns. So every function and the module have docstrings, and `help()` displays them.

**What is the difference between a docstring and a `#` comment? When do I use each?**
(Answer: a docstring is a string placed right under `def` (or at the top of the module). Python keeps it, so `help()` and editors can show it, and it explains what the code does for the people who use it. A `#` comment is ignored by Python and is only for people reading the source. I use docstrings to document functions, classes and modules, and comments to explain why a particular line is written the way it is.)

**Why write the summary in the imperative ("Return") and not the descriptive ("Returns")?**
(Answer: it is the convention in PEP 257. The summary reads like a command describing what calling the function does, the same way a Git commit message does, and it keeps all summaries consistent.)

**What is the difference between Google style and NumPy style docstrings?**
(Answer: both document the same things (arguments, return values), but the layout differs. Google style uses short headings like `Args:` and `Returns:` with indented entries, and is compact. NumPy style uses longer headings underlined with dashes, like `Parameters` followed by `----------`, and is common in scientific libraries such as NumPy and pandas.)

## Task 03: Add type hints and run mypy
**What I did:** Added type hints to every function in `mathutils.py`, ran `mypy` to check them, deliberately introduced a type error to see how mypy reports it, then fixed it.

**Logic of the hints:**
- `nums: list[float]` says the argument is a list of numbers, and `-> float` says the function returns a number.
- `biggest` returns the biggest number, but for an empty list it returns `None`. So its return type is `Optional[int]`, meaning "an int or None".
- `is_prime(n: int) -> bool` takes a whole number and returns `True` or `False`.
- Type hints don't change how the code runs. `mypy` is a separate tool that reads them and checks that the code is consistent with them.

In [32]:
import sys
!"{sys.executable}" -m mypy mathutils.py

Success: no issues found in 1 source file


mypy flagged nothing, because every return value matches its hint. `return 0` is accepted for `float`, since mypy treats an int as valid where a float is expected. `return None` in `biggest` is only allowed because I wrote `Optional[int]`.

In [33]:
from pathlib import Path

p = Path("mathutils.py")
original = p.read_text()
p.write_text(original.replace("return 0", 'return "zero"'))
print("average now returns a string on purpose")

average now returns a string on purpose


In [34]:
!"{sys.executable}" -m mypy mathutils.py

mathutils.py:14: error: Incompatible return value type (got "str", expected "float")  [return-value]
Found 1 error in 1 file (checked 1 source file)


**How mypy reports the mismatch:**
- `mathutils.py:16` is the file and line number of the problem.
- `Incompatible return value type (got "str", expected "float")` says the function is hinted to return a float, but this line returns a string.
- `[return-value]` is the name of the check that failed.
- The last line counts the errors: 1 error in 1 file.
The code would still run, because Python doesn't enforce hints. Only mypy catches this.

In [35]:
p.write_text(original)
print("mathutils.py is fixed again")

mathutils.py is fixed again


In [36]:
!"{sys.executable}" -m mypy mathutils.py

Success: no issues found in 1 source file


In [37]:
import importlib
import mathutils

importlib.reload(mathutils)
print(mathutils.biggest(["a", "c", "b"]))

c


**Do type hints stop me from passing the wrong type at runtime? If not, what is the point of them?**
(Answer: no. `biggest(["a", "c", "b"])` ran fine even though the hint says `list[int]`. Their point is to document what a function expects and returns, and to let tools like mypy and the editor catch mismatches before the code runs.)

**What does `Optional[int]` mean, and when do I need it?**
(Answer: it means "an int or None". I need it when a function can return `None` (or accept `None`), like `biggest` for an empty list. Without it, mypy reports an error on `return None`.)

**What is the difference between what ruff checks and what mypy checks?**
(Answer: ruff is a linter. It looks at the code's style and common mistakes, like unused imports or an undefined variable, without looking at types. mypy checks types: whether the values I pass and return match the hints.)

## Task 04: Format and lint
**What I did:** Made `mathutils.py` badly formatted on purpose and let `black` fix it, checked the change with `git diff`, added unused imports and let `ruff` find and remove them, then introduced an undefined variable to see ruff catch a real bug.

**Logic:**
- `black` is a formatter. It rewrites the layout (spaces, blank lines) into one standard style and never changes what the code does.
- `ruff` is a linter. It reads the code without running it and reports problems, like unused imports or a variable that was never defined. `ruff check --fix` repairs the ones it can fix safely.
- `git diff` shows exactly which lines changed. To see what black changed, I first stage the messy file with `git add`, so Git remembers the messy version, then run black and compare.

In [38]:
import sys
from pathlib import Path

p = Path("mathutils.py")
formatted = p.read_text()

messy = (
    formatted.replace("sum(nums) / len(nums)", "sum(nums)/len(nums)")
    .replace("n < 2", "n<2")
    .replace("n % i == 0", "n%i==0")
    .replace("\n\n\ndef", "\n\n\n\n\n\ndef")
)
p.write_text(messy)
!git add mathutils.py
print("mathutils.py is now messy, and staged in Git")

mathutils.py is now messy, and staged in Git


In [39]:
!"{sys.executable}" -m black mathutils.py

reformatted mathutils.py

All done! ✨ 🍰 ✨
1 file reformatted.


In [40]:
!git diff mathutils.py

diff --git a/Day5/mathutils.py b/Day5/mathutils.py
index ff0c5d3..6c435e1 100644
--- a/Day5/mathutils.py
+++ b/Day5/mathutils.py
@@ -1,9 +1,6 @@
 """Provide small math helpers: average, biggest and is_prime."""
 
 
-
-
-
 def average(nums: list[float]) -> float:
     """Return the mean of a list of numbers.
 
@@ -15,10 +12,7 @@ def average(nums: list[float]) -> float:
     """
     if not nums:
         return 0
-    return sum(nums)/len(nums)
-
-
-
+    return sum(nums) / len(nums)
 
 
 def biggest(nums: list[int]) -> int | None:
@@ -35,9 +29,6 @@ def biggest(nums: list[int]) -> int | None:
     return max(nums)
 
 
-
-
-
 def is_prime(n: int) -> bool:
     """Return whether a number is prime.
 
@@ -48,9 +39,9 @@ def is_prime(n: int) -> bool:
         True if n is prime, otherwise False. Numbers below 2 (0, 1 and
         negatives) are never prime, so they return False.
     """
-    if n<2:
+    if n < 2:
         return False
     for i in range(2, int(n**0.5) + 1):
-        if n%i

In [41]:
!"{sys.executable}" -m black mathutils.py

All done! ✨ 🍰 ✨
1 file left unchanged.


**What black changed:** it removed the extra blank lines (red `-` lines in the diff) and added spaces around `/`, `<`, `%` and `==` (the `+` lines). The second run changed nothing, so black is stable: once a file is formatted, running black again leaves it alone.

In [42]:
formatted = p.read_text()
with_imports = formatted.replace(
    '"""\n\n\ndef average',
    '"""\n\nimport os\nimport sys\nimport json\n\n\ndef average',
    1,
)
p.write_text(with_imports)
print("added three unused imports")

added three unused imports


In [43]:
!"{sys.executable}" -m ruff check mathutils.py

I001 [*] Import block is un-sorted or un-formatted
 --> mathutils.py:3:1
  |
1 |   """Provide small math helpers: average, biggest and is_prime."""
2 |
3 | / import os
4 | | import sys
5 | | import json
  | |___________^
help: Organize imports
  |
2 |
3 + import json
4 | import os
5 | import sys
  - import json
6 |
  |

F401 [*] `os` imported but unused
 --> mathutils.py:3:8
  |
1 | """Provide small math helpers: average, biggest and is_prime."""
2 |
3 | import os
  |        ^^
4 | import sys
5 | import json
  |
help: Remove unused import: `os`
  |
2 |
  - import os
3 | import sys
  |

F401 [*] `sys` imported but unused
 --> mathutils.py:4:8
  |
3 | import os
4 | import sys
  |        ^^^
5 | import json
  |
help: Remove unused import: `sys`
  |
3 | import os
  - import sys
4 | import json
  |

F401 [*] `json` imported but unused
 --> mathutils.py:5:8
  |
3 | import os
4 | import sys
5 | import json
  |        ^^^^
help: Remove unused import: `json`
  |
4 | import sys
  - import json
5

In [44]:
!"{sys.executable}" -m ruff check --fix mathutils.py
!"{sys.executable}" -m black mathutils.py

Found 3 errors (3 fixed, 0 remaining).


reformatted mathutils.py

All done! ✨ 🍰 ✨
1 file reformatted.


`ruff check --fix` removed the three unused imports automatically. It left extra blank lines at the top where they were, so `black` reformatted the file again. The two tools do different jobs, so I run both: ruff to fix problems, black to fix layout.

In [45]:
formatted = p.read_text()
p.write_text(formatted.replace("if n < 2:", "if n < minimum:"))
!"{sys.executable}" -m ruff check mathutils.py

F821 Undefined name `minimum`
  --> mathutils.py:42:12
   |
40 |         negatives) are never prime, so they return False.
41 |     """
42 |     if n < minimum:
   |            ^^^^^^^
43 |         return False
44 |     for i in range(2, int(n**0.5) + 1):
   |

Found 1 error.


ruff reported `F821 Undefined name 'minimum'` with the exact file, line and column, without running the program. This bug would only crash when `is_prime` is called, but ruff found it just by reading the code.

In [46]:
p.write_text(formatted)
!"{sys.executable}" -m black --check mathutils.py
!"{sys.executable}" -m ruff check mathutils.py
!"{sys.executable}" -m pytest -q

All done! ✨ 🍰 ✨
1 file would be left unchanged.


All checks passed!
............                                                             [100%]
12 passed in 0.02s


**What does black change that ruff does not, and vice versa?**
(Answer: black changes layout only (spacing, blank lines, line breaks, quote style) and never changes what the code does. ruff finds problems in the code, like unused imports, undefined names and likely bugs, and can fix some of them. In this task black fixed the spacing and ruff fixed the imports.)

**Why is having one canonical format valuable for a team beyond looking neat?**
(Answer: nobody argues about style in reviews, and diffs only show real changes instead of whitespace noise. That also means fewer merge conflicts, and everyone can read each other's code easily.)

**What kinds of bugs can ruff catch without running my code?**
(Answer: undefined variables (like `minimum`), unused imports and variables, syntax errors, and some common mistakes in how code is written. It reads the code without running it, so it can find bugs in paths that tests never reach.)

## Task 05: Automate it with pre-commit hooks
**What I did:** Created a `.pre-commit-config.yaml` in the repo root with hooks for `black` and `ruff`, installed the hooks into Git, then made a deliberately bad commit and watched it get blocked, fixed it, and committed successfully.

**Logic:**
- A Git hook is a small program Git runs automatically at a certain moment. `pre-commit` is a tool that manages them.
- The config file lists which tools to run. `pre-commit install` copies a hook into `.git/hooks/pre-commit`, so those tools run before every `git commit`.
- If any tool fails, or changes a file, the commit is stopped. It only goes through when every hook passes.
- Hooks run only on the files that are staged for the commit.

In [47]:
%%writefile ../.pre-commit-config.yaml
repos:
  - repo: https://github.com/psf/black
    rev: 24.10.0
    hooks:
      - id: black
  - repo: https://github.com/astral-sh/ruff-pre-commit
    rev: v0.8.0
    hooks:
      - id: ruff

Writing ../.pre-commit-config.yaml


**How the config works:** each `repo:` entry is a tool's repository, `rev:` is the exact version to use, and `id:` is which hook from that repo to run. So `black` reformats my code and `ruff` checks it for problems. Pinning the version means everyone on the team runs the same version of each tool.

In [48]:
%%writefile bad_example.py
import os
def add_one(x):
    return x+1
print( add_one(2) )

Writing bad_example.py


**The bad commit was blocked:**
- `black` reported `Failed` and `files were modified by this hook`: it reformatted my file (spacing, blank lines), and a hook that changes a file always blocks the commit, so I can review the change.
- `ruff` reported `Failed` with `F401 imported but unused` for `import os`. It found a problem it cannot safely fix on its own here, so I must fix it.
- No commit was created.

**The clean commit passed:** after I removed the unused import and let black reformat the file, both hooks showed `Passed` and the commit was created. Commit 1 was blocked and commit 2 went through, which is exactly the behaviour the task asks for.

**What just happened and why it is powerful (cheatsheet):**
I wrote bad code and tried to commit it. Git ran the hooks automatically: black reformatted the file, ruff found an unused import, and the commit was blocked until everything was clean. I did not have to remember to run any tool. This means bad formatting and simple bugs can never reach the repository, so reviews are about real logic.

**What is a Git hook? What other moments can hooks fire on besides commit?**
(Answer: a hook is a script Git runs automatically at a certain moment. Besides `pre-commit`, there are hooks for `commit-msg` (checking the message), `pre-push` (before pushing), `post-merge`, `post-checkout` and others.)

**Why block a bad commit instead of just warning?**
(Answer: warnings get ignored, and once bad code is committed it's in the history and can reach teammates. Blocking catches the problem at the cheapest moment, right where it was made, and makes the standard automatic.)

**What happens when a teammate clones my repo? Do they get my hooks automatically?**
(Answer: no. Hooks live in `.git/hooks`, which is not copied by `git clone`. The config file `.pre-commit-config.yaml` is cloned, but each person must run `pre-commit install` once. That's why the README should say so.)

## Task 06: Refactor a messy notebook into a clean module (Week 1 deliverable)
**What I did:** Took the marks logic from my Day 2 notebook (the `Student` methods and the debugged `class_average` script) and turned it into a clean module, `student_utils.py`, with a test suite, README, and passing black, ruff and mypy checks. The pre-commit hooks from Task 05 run on every commit.

**What I kept and what I dropped:**
- Kept: the maths (average, highest, lowest, pass rate), and the fix that a mark equal to the pass mark counts as a pass.
- Dropped: demo `print` lines, the printing experiments with `__str__` and `__repr__`, the iterator and file-reading experiments, and the debugging scaffolding, because they were exploration and not reusable logic.

**Design:** each function does one thing, takes its input as arguments, and returns a value instead of printing. That makes them easy to test and to reuse.

In [49]:
%%writefile student_utils.py
"""Provide small helpers for analysing a student's marks."""


def average(marks: list[float]) -> float | None:
    """Return the mean of a list of marks.

    Args:
        marks: A list of marks. May be empty.

    Returns:
        The mean mark, or None if the list is empty.
    """
    if not marks:
        return None
    return sum(marks) / len(marks)


def highest(marks: list[float]) -> float | None:
    """Return the highest mark in a list.

    Args:
        marks: A list of marks. May be empty.

    Returns:
        The highest mark, or None if the list is empty.
    """
    return max(marks, default=None)


def lowest(marks: list[float]) -> float | None:
    """Return the lowest mark in a list.

    Args:
        marks: A list of marks. May be empty.

    Returns:
        The lowest mark, or None if the list is empty.
    """
    return min(marks, default=None)


def pass_rate(marks: list[float], passing: float = 40) -> float:
    """Return the percentage of marks that reach the pass mark.

    A mark equal to the pass mark counts as a pass.

    Args:
        marks: A list of marks. May be empty.
        passing: The lowest mark that counts as a pass. Defaults to 40.

    Returns:
        The pass rate from 0.0 to 100.0, or 0.0 if the list is empty.
    """
    if not marks:
        return 0.0
    passed = sum(1 for mark in marks if mark >= passing)
    return passed / len(marks) * 100


def summarise(marks: list[float], passing: float = 40) -> dict[str, float | None]:
    """Return a summary of a list of marks.

    Args:
        marks: A list of marks. May be empty.
        passing: The lowest mark that counts as a pass. Defaults to 40.

    Returns:
        A dict with the keys count, average, highest, lowest and pass_rate.
        For an empty list, count is 0, pass_rate is 0.0 and the other
        values are None.
    """
    return {
        "count": len(marks),
        "average": average(marks),
        "highest": highest(marks),
        "lowest": lowest(marks),
        "pass_rate": pass_rate(marks, passing),
    }

Writing student_utils.py


**How the functions work:**
- `average`, `highest`, `lowest`: each handles the empty list first and returns `None`, because the maths would crash (`max([])`) or divide by zero. `max(marks, default=None)` does that in one step.
- `pass_rate`: counts the marks that are `>= passing` and turns that into a percentage. It uses `>=`, so a mark exactly at the pass mark passes. That is the bug I found with the debugger in Day 2 Task 08 (`>` left out students on the pass mark). Empty list gives 0.0.
- `summarise`: combines the other functions into one dict, like the `calculate_stats` function from Day 1. Reusing the small functions keeps it short.
- Every function has type hints and a Google-style docstring, and the module has a one-line docstring.

In [50]:
%%writefile test_student_utils.py
from student_utils import average, highest, lowest, pass_rate, summarise

MARKS = [88, 92, 79, 40]


def test_average_basic():
    assert average(MARKS) == 74.75


def test_average_empty():
    assert average([]) is None


def test_average_single():
    assert average([55]) == 55


def test_average_floats():
    assert average([1.5, 2.5]) == 2.0


def test_highest_basic():
    assert highest(MARKS) == 92


def test_highest_empty():
    assert highest([]) is None


def test_highest_all_equal():
    assert highest([70, 70, 70]) == 70


def test_lowest_basic():
    assert lowest(MARKS) == 40


def test_lowest_empty():
    assert lowest([]) is None


def test_pass_rate_all_pass():
    assert pass_rate(MARKS) == 100.0


def test_pass_rate_exactly_at_pass_mark():
    assert pass_rate([40]) == 100.0


def test_pass_rate_none_pass():
    assert pass_rate([10, 20, 30]) == 0.0


def test_pass_rate_mixed():
    assert pass_rate([88, 92, 79, 30]) == 75.0


def test_pass_rate_empty():
    assert pass_rate([]) == 0.0


def test_pass_rate_custom_pass_mark():
    assert pass_rate([50, 60], passing=55) == 50.0


def test_summarise_basic():
    assert summarise(MARKS) == {
        "count": 4,
        "average": 74.75,
        "highest": 92,
        "lowest": 40,
        "pass_rate": 100.0,
    }


def test_summarise_empty():
    assert summarise([]) == {
        "count": 0,
        "average": None,
        "highest": None,
        "lowest": None,
        "pass_rate": 0.0,
    }

Writing test_student_utils.py


| Function | Happy path | Edge cases |
|---|---|---|
| `average` | the Day 2 marks give 74.75, and floats | empty list, a single mark |
| `highest` | the Day 2 marks give 92 | empty list, all marks equal |
| `lowest` | the Day 2 marks give 40 | empty list |
| `pass_rate` | all pass gives 100.0, mixed gives 75.0, custom pass mark | a mark exactly at the pass mark (the Day 2 bug), nobody passes, empty list |
| `summarise` | the full dict for the Day 2 marks | the dict for an empty list |

The test `test_pass_rate_exactly_at_pass_mark` is a regression test: if anyone changes `>=` back to `>`, it fails straight away.

In [51]:
%%writefile README.md
# student_utils

Small, typed helpers for analysing a student's marks: average, highest, lowest,
pass rate and a one-call summary. Refactored from my Day 2 notebook.

## Functions

- `average(marks)`: the mean mark, or `None` for an empty list
- `highest(marks)` and `lowest(marks)`: the top and bottom mark, or `None` for an empty list
- `pass_rate(marks, passing=40)`: the percentage of marks at or above the pass mark (a mark equal to the pass mark passes), or 0.0 for an empty list
- `summarise(marks, passing=40)`: a dict with `count`, `average`, `highest`, `lowest` and `pass_rate`

## Example

    from student_utils import summarise

    print(summarise([88, 92, 79, 40]))
    # {'count': 4, 'average': 74.75, 'highest': 92, 'lowest': 40, 'pass_rate': 100.0}

## Setup (Python 3.10 or newer)

From the repository root (on Mac or Linux, activate with `source .venv/bin/activate`):

    python -m venv .venv
    .venv\Scripts\activate
    pip install -r requirements.txt
    pre-commit install

## Run the tests

    cd Day5
    pytest -v

## Check code quality

    black --check .
    ruff check . --extend-exclude "*.ipynb"
    mypy .

Writing README.md


In [52]:
%%writefile ../.pre-commit-config.yaml
repos:
  - repo: https://github.com/psf/black
    rev: 24.10.0
    hooks:
      - id: black
  - repo: https://github.com/astral-sh/ruff-pre-commit
    rev: v0.8.0
    hooks:
      - id: ruff
        exclude: '\.ipynb$'

Overwriting ../.pre-commit-config.yaml


In [53]:
import sys

!"{sys.executable}" -m pytest -v
!"{sys.executable}" -m black --check .
!"{sys.executable}" -m ruff check . --extend-exclude "*.ipynb"
!"{sys.executable}" -m mypy .

============================= test session starts =============================
platform win32 -- Python 3.12.1, pytest-9.2.0, pluggy-1.7.0 -- c:\Users\gajen\OneDrive\Desktop\Codetrade_internship\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\gajen\OneDrive\Desktop\Codetrade_internship\Day5
plugins: platformdirs-4.14.7
collecting ... collected 29 items

test_mathutils.py::test_average_basic PASSED                             [  3%]
test_mathutils.py::test_average_empty PASSED                             [  6%]
test_mathutils.py::test_average_single PASSED                            [ 10%]
test_mathutils.py::test_biggest_basic PASSED                             [ 13%]
test_mathutils.py::test_biggest_empty PASSED                             [ 17%]
test_mathutils.py::test_biggest_negatives PASSED                         [ 20%]
test_mathutils.py::test_is_prime_two PASSED                              [ 24%]
test_mathutils.py::test_is_prime_seven PASSED                   

Skipping .ipynb files as Jupyter dependencies are not installed.
You can fix this by running ``pip install "black[jupyter]"``
All done! ✨ 🍰 ✨
5 files would be left unchanged.


All checks passed!
Success: no issues found in 5 source files


**What logic from my notebook is worth keeping, and what was just exploration?**
(Answer: the marks maths (average, highest, lowest, pass rate) and the fix that a mark on the pass mark passes are worth keeping. The prints, the `__str__` and `__repr__` experiments, the iterator and file-reading experiments and the debugging scaffolding were exploration.)

**Which edge cases in my code are most likely to break? Those need tests most.**
(Answer: the empty list, where `max([])` crashes and dividing by zero fails; a list with one mark; and a mark exactly at the pass mark, where `>` and `>=` give different answers. All three have tests.)

**If a teammate opened this repo cold, could they run it from my README alone?**
(Answer: yes. The README lists the Python version, how to create and activate the environment, install the requirements, install the hooks, run the tests, and run the quality checks.)

**Week 1 deliverable checklist:**
1. Clean module: `student_utils.py` (small functions, type hints, docstrings)
2. Test suite: `test_student_utils.py` (17 tests, happy paths and edge cases)
3. Green tooling: black, ruff and mypy all pass
4. Pre-commit hooks: `.pre-commit-config.yaml` active, and a bad commit was blocked in Task 05
5. README: `README.md` says what it does and how to run the tests